# 02. Análise exploratória dos dados (tiny)

Uma plataforma liga pequenos produtores do distrito de Braga aos clientes através de um centro de
recolha com frota própria. Cada produto é recolhido no produtor, passa pelo armazém e é entregue no
local escolhido pelo cliente, sempre dentro de janelas temporais. O agente do projeto planeia essas
rotas com o menor número de km.

Antes de modelar, olhamos para o dataset `tiny`, com algumas comparações com o `small`: que tabelas
existem, se os dados estão íntegros, quanto pesa a procura face à capacidade, como são as janelas
temporais e onde ficam as paragens. A última secção repete as verificações principais no `small`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.model import load_problem, read_tables, audit, plot_locations

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
ROOT = Path.cwd().parent
tables = {name: read_tables(ROOT / "data" / name) for name in ("tiny", "small")}
tiny = tables["tiny"]


def hours(delta):
    return delta.dt.total_seconds() / 3600

## 1. Tabelas

Cada dataset tem 12 CSV com a mesma estrutura. A tabela compara o número de linhas de cada um.

In [ ]:
pd.DataFrame({name: {table: len(df) for table, df in dataset.items()} for name, dataset in tables.items()})

In [ ]:
tiny["rules"]

As regras fixam o fluxo produtor → armazém → local de entrega, 10 min de serviço em cada paragem, 15 min
para descarregar e 15 para carregar no armazém, e no máximo 24 h de permanência lá. As janelas limitam a
hora a que o serviço começa. A matriz de distâncias é sintética: distância em linha reta × 1,3,
percorrida a 45 km/h.

In [ ]:
tiny["vehicles"], tiny["collection_centers"]

## 2. Integridade

A auditoria procura valores em falta, duplicados, chaves primárias repetidas, chaves estrangeiras
partidas, locais que faltam na tabela `locations`, coordenadas inválidas, janelas trocadas e pesos que
não batem certo com `quantity × unit_weight_kg`. Na matriz de distâncias confirma que existem todos os
pares, sem valores negativos, com tipos coerentes e simetria. Verifica ainda se alguma linha foi
encomendada depois de abrir a janela de recolha.

In [ ]:
audit_results = pd.concat([audit(dataset).assign(dataset=name) for name, dataset in tables.items()])
audit_results.pivot_table(index="status", columns="dataset", values="check", aggfunc="count", fill_value=0)

In [ ]:
audit_results[audit_results.status != "ok"]

Não há erros nem avisos em nenhum dos datasets, só duas observações: há pares de locais de entrega com
as mesmas coordenadas (distância 0) e nenhuma linha foi encomendada depois de a janela de recolha abrir.
Mesmo assim, o modelo impõe `recolha ≥ order_datetime`, porque é o que faz sentido no negócio.

## 3. Procura e capacidade

In [ ]:
items = tiny["order_items"].merge(tiny["products"][["product_id", "producer_id", "name", "category"]], on="product_id")
items = items.merge(tiny["orders"][["order_id", "delivery_location_id", "order_datetime"]], on="order_id")
for column in ("availability_start", "availability_end", "delivery_start", "delivery_end", "order_datetime"):
    items[column] = pd.to_datetime(items[column])
items["pickup_day"] = items.availability_start.dt.date
items["delivery_day"] = items.delivery_start.dt.date
print(f"linhas: {len(items)}, encomendas: {items.order_id.nunique()}, peso total: {items.weight_kg.sum():.1f} kg")
print("capacidade da frota (uma viagem cada):", tiny["vehicles"].capacity_kg.sum(), "kg")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
items.pivot_table(index="producer_id", columns="pickup_day", values="weight_kg", aggfunc="sum").plot.bar(ax=axes[0])
axes[0].set_title("kg a recolher por produtor e dia")
items.pivot_table(index="delivery_location_id", columns="delivery_day", values="weight_kg",
                  aggfunc="sum").plot.bar(ax=axes[1])
axes[1].set_title("kg a entregar por local e dia")
items.groupby("order_id").weight_kg.sum().sort_values().plot.barh(ax=axes[2])
for capacity in tiny["vehicles"].capacity_kg:
    axes[2].axvline(capacity, color="red", ls="--")
axes[2].set_title("peso por encomenda (linhas: capacidade)")
fig.tight_layout()

In [ ]:
pd.crosstab(items.pickup_day, items.delivery_day, values=items.weight_kg, aggfunc="sum").round(1)

Nenhuma encomenda chega perto da capacidade de um veículo (150 e 200 kg). A capacidade só conta quando
se juntam várias recolhas ou entregas na mesma viagem.

No tiny, os 270 kg do horizonte cabiam numa só volta da frota (150 + 200 = 350 kg). O aperto está no
dia 31: há 165 kg para entregar num intervalo curto, e 62 kg deles foram recolhidos no dia 30 e passam a
noite no armazém, sujeitos ao limite de 24 h. No `small`, o peso total (580 kg) já passa a
capacidade da frota (550 kg), por isso são precisas várias viagens.

## 4. Janelas temporais

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))
by_delivery = items.sort_values(["delivery_start", "order_id"]).reset_index(drop=True)
for i, line in by_delivery.iterrows():
    ax.plot([line.availability_start, line.availability_end], [i, i], color="tab:green", lw=3)
    ax.plot([line.delivery_start, line.delivery_end], [i, i], color="tab:blue", lw=3)
ax.set_yticks(range(len(by_delivery)),
              [f"enc {line.order_id} / linha {line.order_item_id} (P{line.producer_id})" for line in by_delivery.itertuples()],
              fontsize=6)
ax.set_title("Janela de recolha (verde) e de entrega (azul) de cada linha")
fig.tight_layout()

In [ ]:
pd.DataFrame({
    "janela recolha (h)": hours(items.availability_end - items.availability_start),
    "janela entrega (h)": hours(items.delivery_end - items.delivery_start),
    "folga recolha->entrega (h)": hours(items.delivery_end - items.availability_start),
}).describe().round(2)

In [ ]:
# linhas recolhidas num dia e entregues no seguinte: passam a noite no armazém
overnight = items[items.pickup_day != items.delivery_day]
print(len(overnight), "linhas atravessam a noite;", overnight.order_id.nunique(), "encomendas:",
      sorted(overnight.order_id.unique()))
gap = hours(overnight.delivery_end - overnight.availability_start)
print(f"tempo entre abrir a recolha e fechar a entrega: {gap.min():.1f}–{gap.max():.1f} h (limite de permanência: 24 h)")

As janelas de recolha caem todas entre as 06:45 e as 12:15, e as de entrega entre as 07:15 e as 14:45:
recolhe-se de manhã e entrega-se a meio do dia. Dentro de cada encomenda, todas as linhas têm a mesma
janela de entrega, por isso faz sentido tratar a entrega de uma encomenda como um único pedido.

Três encomendas (2, 8 e 9) são recolhidas no dia 30 e entregues no dia 31. Entre abrir a recolha e
fechar a entrega passam 25 a 29 h, mais do que as 24 h permitidas no armazém. Para estas encomendas,
recolher muito cedo ou entregar muito tarde viola a permanência máxima.

## 5. Geografia e distâncias

In [ ]:
tiny_problem = load_problem(ROOT / "data" / "tiny")
ax = plot_locations(tiny_problem)
ax.set_title("Paragens do dataset tiny");

In [ ]:
distances = tiny["distance_matrix"].pivot(index="origin_location_id", columns="destination_location_id",
                                          values="distance_km")
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(distances.fillna(0), cmap="viridis")
ax.set_xticks(range(len(distances)), distances.columns)
ax.set_yticks(range(len(distances)), distances.index)
for i in range(len(distances)):
    for j in range(len(distances)):
        km = distances.iloc[i, j]
        ax.text(j, i, "" if pd.isna(km) else f"{km:.0f}", ha="center", va="center", fontsize=7, color="white")
fig.colorbar(image, label="km")
ax.set_title("Matriz de distâncias (km)");

In [ ]:
matrix = tiny["distance_matrix"]
speed = (matrix.distance_km / matrix.duration_min * 60).round(1)
print("velocidade implícita (km/h):", speed.describe()[["min", "max"]].to_dict())
depot = tiny_problem.depot
print("distância do centro (km):", {loc: tiny_problem.d(depot, loc) for loc in sorted(distances.index) if loc != depot})

O centro de recolha, em Braga, fica entre os produtores, a leste (Fafe, Vieira do Minho, Terras de
Bouro), e os clientes, a oeste e a sul (Esposende, Famalicão, Guimarães). Cada local está a 19–38 km do
centro e todas as viagens começam e acabam lá, por isso o número de viagens pesa muito no custo. Os
locais 7 e 9 (Casa 3 e Loja 5, em Guimarães) estão no mesmo ponto.

## 6. Pedidos depois do agrupamento

In [ ]:
rows = [dict(id=request_id, tipo=request.kind, local=request.loc, inicio=tiny_problem.clock(request.tw[0]),
             fim=tiny_problem.clock(request.tw[1]), kg=request.weight, linhas=len(request.items),
             depende_de=",".join(request.deps))
        for request_id, request in tiny_problem.requests.items()]
pd.DataFrame(rows)

Juntando as linhas do mesmo produtor cujas janelas se intersetam, as 40 linhas do tiny dão 6 recolhas
(3 produtores × 2 dias) e 12 entregas (uma por encomenda): 18 pedidos ao todo. As janelas das recolhas
encolhem para a interseção; a P1, por exemplo, fica só com 15 min.

## 7. Comparação com o small

O modelo trata os dois datasets da mesma forma. A tabela repete no small as verificações de que saíram
as decisões de modelação.

In [ ]:
def check_assumptions(name):
    lines = tables[name]["order_items"].copy()
    for column in ("availability_start", "availability_end", "delivery_start", "delivery_end"):
        lines[column] = pd.to_datetime(lines[column])
    problem = load_problem(ROOT / "data" / name)
    crosses_midnight = ((lines.availability_start.dt.date != lines.availability_end.dt.date)
                        | (lines.delivery_start.dt.date != lines.delivery_end.dt.date))
    overnight = lines[lines.availability_start.dt.date != lines.delivery_start.dt.date]
    gap = hours(overnight.delivery_end - overnight.availability_start)
    kinds = pd.Series([request.kind for request in problem.requests.values()]).value_counts()
    return {
        "janelas de entrega por encomenda (máx.)": lines.groupby("order_id")[["delivery_start", "delivery_end"]].nunique().max().max(),
        "linhas com janela a passar a meia-noite": int(crosses_midnight.sum()),
        "janelas de recolha": f"{lines.availability_start.dt.time.min():%H:%M}–{lines.availability_end.dt.time.max():%H:%M}",
        "janelas de entrega": f"{lines.delivery_start.dt.time.min():%H:%M}–{lines.delivery_end.dt.time.max():%H:%M}",
        "encomenda mais pesada (kg)": round(lines.groupby("order_id").weight_kg.sum().max(), 1),
        "capacidades (kg)": " + ".join(f"{c:.0f}" for c in tables[name]["vehicles"].capacity_kg),
        "encomendas que passam a noite": overnight.order_id.nunique(),
        "abrir recolha → fechar entrega (h)": f"{gap.min():.1f}–{gap.max():.1f}",
        "linhas → recolhas + entregas": f"{len(lines)} → {kinds['P']} + {kinds['D']}",
        "distância ao centro (km)": f"{min(problem.d(problem.depot, r.loc) for r in problem.requests.values()):.0f}–"
                                    f"{max(problem.d(problem.depot, r.loc) for r in problem.requests.values()):.0f}",
    }

pd.DataFrame({name: check_assumptions(name) for name in ("tiny", "small")})

No small, cada encomenda também tem uma só janela de entrega, nenhuma janela passa a meia-noite e
nenhuma encomenda sozinha chega perto da capacidade. Há 6 encomendas que passam a noite no armazém,
contra 3 no tiny, e numa delas há só 23,8 h entre abrir a recolha e fechar a entrega, o que fica abaixo
do limite de 24 h. As janelas são mais largas (recolhas entre as 06:30 e as 14:15, entregas até às
18:15) e um dos locais fica no próprio centro, à distância 0.

## 8. Conclusões

Os dados estão íntegros. As restrições que mais apertam são as temporais: as janelas e o encadeamento
recolha → armazém → entrega. Como todos os trajetos passam pelo centro, o número de viagens domina o
custo. Daqui saem duas decisões de modelação: cada encomenda é entregue numa só paragem, e as recolhas
agrupam-se por produtor e janela.